# tensor-reshape-view — ex1: choose reshape vs view based on whether the input is contiguous

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `tensor-reshape-view`. Running the final beacon cell reports progress against the `PyTorch: reshape vs view` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: reshape vs view` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`tensor-reshape-view`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "tensor-reshape-view"
DD_SUBTOPIC = "PyTorch: reshape vs view"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## PyTorch: `.reshape()` vs `.view()` — quick refresher

Both change shape without changing data. They differ on what happens when the source is NOT contiguous in memory:

```python
x = torch.arange(12).reshape(3, 4)
x.is_contiguous()         # True
x.view(4, 3)              # OK — same storage, new strides
x.reshape(4, 3)           # OK — same storage, new strides

y = x.T                    # transpose; y.is_contiguous() == False
y.view(12)                 # RuntimeError: view requires contiguous
y.reshape(12)              # OK — silently copies into a new contiguous tensor
```

**`.view()` is strict.** Requires the source to be contiguous. Returns a view (shares storage). Cheap (O(1)). Raises `RuntimeError` if the source isn't contiguous.

**`.reshape()` is forgiving.** First tries to return a view (O(1)). If that's not possible (non-contiguous source), it silently makes a contiguous copy (O(n)). Always succeeds for shape-compatible targets.

**Decision tree.**
- Need to GUARANTEE no copy → `.view()` (and call `.contiguous()` first if you're not sure).
- Don't care about the copy → `.reshape()` (default in most code).
- Want explicit copy → `.reshape().clone()` or `.contiguous().view()`.

**Why ARENA's stride exercises insist on `.view()`.** The exercise is teaching stride math — accidentally copying defeats the lesson. In application code, `.reshape()` is the safer default.

### Exercise 1 — choose reshape vs view based on whether the input is contiguous

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the reshape-vs-view decision rule: use `.view()` when you require a no-copy reshape (and want a `RuntimeError` if that's impossible), use `.reshape()` when a copy is acceptable as a fallback.
> Keywords: reshape, view, contiguous, stride
> ```

**KCs targeted:** `view-requires-contiguous`, `reshape-falls-back-to-copy`

Implement TWO functions that demonstrate the difference between `.view()` and `.reshape()`:

**1. `ex1_strict_view(x, shape)`** — return `x.view(*shape)`. This MUST raise `RuntimeError` when `x` is not contiguous and the target shape requires a memory reorder. Do NOT call `.contiguous()` first.

**2. `ex1_safe_reshape(x, shape)`** — return `x.reshape(*shape)`. This works on contiguous AND non-contiguous inputs (silently copies when needed).

Inputs:
- `x`: a `Tensor`.
- `shape`: a tuple of ints (the target shape).

Output: a `Tensor` of the target shape.

Note: `x.view(*shape)` unpacks the tuple into positional ints (the standard PyTorch idiom). Same for `x.reshape(*shape)`.

In [ ]:
def ex1_strict_view(x: Tensor, shape: tuple) -> Tensor:
    """Return x.view(*shape). Raises on non-contiguous."""
    raise NotImplementedError()

def ex1_safe_reshape(x: Tensor, shape: tuple) -> Tensor:
    """Return x.reshape(*shape). Always succeeds (may copy)."""
    raise NotImplementedError()


def _test_ex1():
    # === On contiguous input, both work and return EQUAL data ===
    x = t.arange(12).reshape(3, 4)
    assert x.is_contiguous()
    v = ex1_strict_view(x, (4, 3))
    r = ex1_safe_reshape(x, (4, 3))
    assert v.shape == (4, 3)
    assert r.shape == (4, 3)
    assert t.equal(v, r), 'on contiguous input, view and reshape produce equal results'

    # === On contiguous input, view returns a VIEW (shares storage) ===
    x = t.arange(12).reshape(3, 4)
    v = ex1_strict_view(x, (12,))
    assert v.data_ptr() == x.data_ptr(), 'view should share storage with source'

    # === On NON-contiguous input, view RAISES ===
    y = x.T   # transpose → non-contig
    assert not y.is_contiguous()
    try:
        ex1_strict_view(y, (12,))
    except RuntimeError:
        pass
    else:
        raise AssertionError('view on non-contig should raise RuntimeError')

    # === On NON-contiguous input, reshape silently copies and succeeds ===
    r = ex1_safe_reshape(y, (12,))
    assert r.shape == (12,), f'expected (12,), got {tuple(r.shape)}'
    # Reshape on non-contig may share or copy; the CONTRACT is just that it works.
    # The data must equal y flattened in row-major.
    expected = y.contiguous().flatten()
    assert t.equal(r, expected), f'reshape result mismatch: got {r} expected {expected}'

    # === 1-D source can be reshaped back into a 1-D target of any compatible length ===
    x_flat = t.tensor([7.0])
    v_sc = ex1_strict_view(x_flat, (1,))
    assert v_sc.shape == (1,), f'expected (1,), got {tuple(v_sc.shape)}'
    assert v_sc.item() == 7.0
    r_sc = ex1_safe_reshape(x_flat, (1, 1, 1))
    assert r_sc.shape == (1, 1, 1)
    assert r_sc.item() == 7.0

    # === reshape can collapse / expand any compatible shape ===
    x = t.arange(24)
    for shp in [(24,), (4, 6), (2, 3, 4), (1, 24, 1)]:
        r = ex1_safe_reshape(x, shp)
        assert r.shape == shp, f'expected {shp}, got {tuple(r.shape)}'
        assert t.equal(r.flatten(), x), f'data lost reshaping to {shp}'

    # === Confirm view did NOT call contiguous internally ===
    # If a student wrote `x.contiguous().view(*shape)`, the non-contig test above
    # would have silently succeeded instead of raising. Re-test to be sure.
    x = t.arange(6).reshape(2, 3)
    y = x.T   # (3, 2) non-contig
    raised = False
    try:
        _ = ex1_strict_view(y, (6,))
    except RuntimeError:
        raised = True
    assert raised, (
        'ex1_strict_view must NOT call .contiguous() — '
        'it must raise on non-contig input'
    )
    _dd_passed.add('ex1')
    print("ex1 ✓")

_test_ex1()

<details><summary>Solution</summary>

```python
def ex1_strict_view(x, shape):
    return x.view(*shape)

def ex1_safe_reshape(x, shape):
    return x.reshape(*shape)
```

**Two one-liners, two different contracts.** `view` is strict, no-copy, and shares storage; `reshape` is permissive and copies when needed. The code is trivial — the LEARNING is which one to reach for at each call site.

**Why ARENA insists on `.view()` for stride exercises.** The lesson is stride arithmetic. If `.reshape()` silently copies, you've lost the stride property the test was checking. `.view()` raises so you notice.

**Application code defaults to `.reshape()`.** Performance-sensitive inner loops use `.view()` (with `.contiguous()` upstream to guarantee it works). Most code just wants the shape change and doesn't care about the copy.

**`.view()` is a bit faster when it works** — no branch, no contiguity check on the copy path. But the gap is measured in nanoseconds; correctness matters more.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()